# ── 0. 配置区 ──────────────────────────────────────────────

In [ ]:
from pathlib import Path

# 输出 CSV 名称（不含扩展名）
EXPORT_CSV_NAME = "ms2_0605_0923"

# 图像根目录列表（取消注释要扫描的目录）
ROOT_LIST = [
    # ms1
    # "/media/tianqi/16tb/SWD/01_Data/2024_SWD_data_RAW/eachFarm/ms1/Desktop_ms1_0706_0605",
    # "/media/tianqi/16tb/SWD/01_Data/2024_SWD_data_RAW/eachFarm/ms1/Desktop_ms1_0730_0707",
    # ms2
    # "/media/tianqi/16tb/SWD/01_Data/2024_SWD_data_RAW/eachFarm/ms2/Desktop_ms2_0706_0605",
    # "/media/tianqi/16tb/SWD/01_Data/2024_SWD_data_RAW/eachFarm/ms2/Desktop_ms2_0730_0705",
    # sw2
    # "/media/tianqi/16tb/SWD/01_Data/2024_SWD_data_RAW/eachFarm/sw2/Desktop_sw2_0730_0706",
    # other
    "/home/tianqi/D/01_Projects/01_swd/02_code/pipeline/ultralytics_ty/_ty/01_data/a04_whole_raw_data/ms2_0605_0923/",
]

# 子路径（可为空字符串，或如 "images/4656x3496_autofocus/"）
SUB_PATH = ""

# 文件名中的年份
YEAR = 2024

# ── 1. 初始化日志 ──────────────────────────────────────────

In [ ]:
import logging
import ipynbname

log_file_name = ipynbname.name() + ".log"
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    handlers=[
        logging.FileHandler(log_file_name, encoding="utf-8"),
        logging.StreamHandler(),
    ],
)
logger = logging.getLogger(__name__)
logger.info("============ Notebook logging 初始化完成 ============")

# ── 2. Step 1：扫描原始数据子目录 ──────────────────────────

In [ ]:
# ── Step 1：扫描 ROOT_LIST 下的子目录 ───────────────────────
# 输入：ROOT_LIST
# 输出：打印子目录列表供 Master 确认

logger.info("Step 1 开始：扫描数据子目录")

for root_str in ROOT_LIST:
    root = Path(root_str)
    subdirs = sorted([p for p in root.iterdir() if p.is_dir()])
    print(f"\n[ROOT] {root}")
    for d in subdirs:
        print(f'    "{d.resolve()}",')

logger.info("Step 1 完成：子目录扫描结束")

# ── 3. Step 2：导入数据，解析文件名 ─────────────────────────

In [ ]:
# ── Step 2：遍历图像文件，解析时间+焦距，收集行数据 ──────────
# 输入：ROOT_LIST 下 .jpg 图像
# 输出：df（含 path/datetime/focal/resolution）

logger.info("Step 2 开始：导入数据，解析文件名")

import pandas as pd
from datetime import datetime
from PIL import Image

rows = []

for root_str in ROOT_LIST:
    for p in Path(root_str).rglob(SUB_PATH + "*.jpg"):
        try:
            mmdd, hhmm, focal = p.stem.split("_")
            dt = datetime(YEAR, int(mmdd[:2]), int(mmdd[2:]), int(hhmm[:2]), int(hhmm[2:]))
            with Image.open(p) as im:
                w, h = im.size
            rows.append({"path": str(p), "datetime": dt, "focal": int(focal), "resolution": f"{w}x{h}"})
        except Exception as e:
            logger.warning(f"文件名格式不符或图像读取失败，跳过: {p.name} ({e})")

df = pd.DataFrame(rows).sort_values("datetime").reset_index(drop=True)
logger.info(f"Step 2 完成：共解析 {len(df)} 张图像")

# ── 4. Step 3：展开分解数据，按日focal汇总，导出 CSV ────────

In [ ]:
# ── Step 3a：展开分解，计算间隔时间，导出点1 CSV ──────────────
# 输入：df
# 输出：{EXPORT_CSV_NAME}.csv

logger.info("Step 3a 开始：展开分解数据")

df1 = df.copy()
df1[['date', 'time']] = df1['datetime'].astype(str).str.split(' ', expand=True)
df1['gap_time'] = df1['datetime'].diff().dt.total_seconds() / 60

try:
    df1.to_csv(f"{EXPORT_CSV_NAME}.csv", index=False)
    logger.info(f"Step 3a 完成：保存 {EXPORT_CSV_NAME}.csv，shape={df1.shape}")
except Exception as e:
    logger.error(f"保存 CSV 失败: {e}")

In [ ]:
# ── Step 3b：按 (date, focal) 汇总每日开始/结束/数量/时长 ────
# 输入：df1
# 输出：df2（daily summary by focal）

logger.info("Step 3b 开始：按日focal汇总")

df2 = df1.copy()
daily_summary = (
    df2.groupby(["date", "focal"])
    .agg(start_time=("time", "first"), end_time=("time", "last"), image_count=("path", "count"))
    .reset_index()
)
start_dt = pd.to_datetime(daily_summary["start_time"], format="%H:%M:%S")
end_dt   = pd.to_datetime(daily_summary["end_time"],   format="%H:%M:%S")
duration = end_dt - start_dt
duration = duration.where(duration >= pd.Timedelta(0), duration + pd.Timedelta(days=1))
sec = duration / pd.Timedelta(seconds=1)
hh  = (sec // 3600).astype(int)
mm  = ((sec % 3600) // 60).astype(int)
daily_summary["duration(hh:mm)"] = hh.astype(str).str.zfill(2) + ":" + mm.astype(str).str.zfill(2)
df2 = daily_summary
logger.info(f"Step 3b 完成：daily_summary shape={df2.shape}")

In [ ]:
# ── Step 3c：填补缺失日期，导出点2 CSV ────────────────────────
# 输入：df2
# 输出：{EXPORT_CSV_NAME}_proc.csv（含连续日期）

logger.info("Step 3c 开始：填补缺失日期")

unique_focal_list = df2['focal'].unique().tolist()
out_parts = []

for focal in unique_focal_list:
    df3 = df2[df2['focal'] == focal].copy()
    df3['date'] = pd.to_datetime(df3['date'], errors='coerce').dt.date
    df3 = df3.sort_values('date')
    all_dates = pd.date_range(start=df3['date'].min(), end=df3['date'].max(), freq='D').date
    df3 = df3.set_index('date').reindex(all_dates).rename_axis('date').reset_index()
    df3["focal"] = focal
    out_parts.append(df3)

df_merged = pd.concat(out_parts, ignore_index=True)

try:
    df_merged.to_csv(f"{EXPORT_CSV_NAME}_proc.csv", index=False)
    logger.info(f"Step 3c 完成：保存 {EXPORT_CSV_NAME}_proc.csv，shape={df_merged.shape}")
except Exception as e:
    logger.error(f"保存 CSV 失败: {e}")

# ── 5. 验证 ────────────────────────────────────────────────

In [ ]:
# ── 验证：抽查最终合并结果 ────────────────────────────────────
from IPython.display import display

logger.info(f"验证：df_merged shape={df_merged.shape}，focal列表={unique_focal_list}")
display(df_merged.head(10))
display(df_merged.tail(5))